# 11 · Harder task: jittered ROI crops (new dataset) + no-network baselines
**Runtime: CPU** (about 2–3 h; resumable). Does not touch any training run or the old crops.

Why: on the original crops (lesion always centred, crop = 1.4 × lesion size) a centred disc with no network reaches Dice 0.86.
Here the SAME 400 CBIS-DDSM ROIs (same patients, same folds) and the SAME DMID masses are re-cropped from the full
mammograms with a random, fixed-per-ROI crop size (1.5–3.0 × the longer lesion side) and a random shift of the lesion
inside the crop (lesion always fully inside). Crops are deterministic (seed derived from the ROI id).

Writes: `data/processed/cbis_mass_256_jit/` (images/, masks/, folds.csv) and `data/external/dmid_mass_256_jit/roi/`
(images/, masks/, metadata.csv), then the mean-mask and centred-disc baselines (`results/p2_jit_baselines.csv`).
**Go / no-go:** continue to training only if the CBIS baselines fall well below 0.86 (target about 0.6 or lower).


In [ ]:
# 1 · Drive + config
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT  = '/content/drive/MyDrive/quantum-placement'
OLD_CBIS = f'{PROJECT}/data/processed/cbis_mass_256'          # read only: subset + folds
NEW_CBIS = f'{PROJECT}/data/processed/cbis_mass_256_jit'
NEW_DMID = f'{PROJECT}/data/external/dmid_mass_256_jit/roi'
OLD_DMID_DEDUP = f'{PROJECT}/results/p2_analysis_v2/dmid_dedup.csv'
RAW_TMP  = '/content/raw_tmp'
CROP_SIZE, SCALE_LO, SCALE_HI, SHIFT = 256, 1.5, 3.0, 0.8     # crop side = U(1.5,3.0) x lesion; shift up to 80% of the slack
for d in [f'{NEW_CBIS}/images', f'{NEW_CBIS}/masks', f'{NEW_DMID}/images', f'{NEW_DMID}/masks', RAW_TMP]:
    os.makedirs(d, exist_ok=True)


In [ ]:
# 2 · Helpers (TCIA access, windowing, mask selection from notebook 01) + jittered crop
!pip -q install pydicom tifffile imagecodecs
import pydicom, hashlib

import os, re, io, json, shutil, zipfile, random, time
import numpy as np
import pandas as pd
import cv2
import requests

TCIA_API = "https://services.cancerimagingarchive.net/nbia-api/services/v1"


# ---------- TCIA access ----------
def get_series_table(collection="CBIS-DDSM", retries=3):
    """All series in the collection, as a DataFrame (one row per series)."""
    for i in range(retries):
        try:
            r = requests.get(f"{TCIA_API}/getSeries", params={"Collection": collection}, timeout=120)
            r.raise_for_status()
            return pd.DataFrame(r.json())
        except Exception as e:
            print(f"  getSeries failed ({e}); retry {i+1}/{retries}")
            time.sleep(5 * (i + 1))
    raise RuntimeError("Could not reach TCIA. Check your internet / try again later.")


def download_series(uid, out_dir, retries=3):
    """Download one series as a zip and extract its .dcm files into out_dir/uid. Returns file list."""
    dest = os.path.join(out_dir, uid)
    if os.path.isdir(dest) and any(f.endswith(".dcm") for f in os.listdir(dest)):
        return sorted(os.path.join(dest, f) for f in os.listdir(dest) if f.endswith(".dcm"))
    os.makedirs(dest, exist_ok=True)
    for i in range(retries):
        try:
            r = requests.get(f"{TCIA_API}/getImage", params={"SeriesInstanceUID": uid}, timeout=600)
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                z.extractall(dest)
            break
        except Exception as e:
            print(f"  download {uid[-12:]} failed ({e}); retry {i+1}/{retries}")
            time.sleep(5 * (i + 1))
    else:
        shutil.rmtree(dest, ignore_errors=True)
        return []
    files = []
    for root, _, fs in os.walk(dest):
        files += [os.path.join(root, f) for f in fs if f.lower().endswith(".dcm")]
    return sorted(files)


# ---------- case table ----------
ROI_PID = re.compile(r"^(Mass-(Training|Test)_P_\d+_(LEFT|RIGHT)_(CC|MLO))_(\d+)$")
FULL_PID = re.compile(r"^Mass-(Training|Test)_P_\d+_(LEFT|RIGHT)_(CC|MLO)$")


def build_case_table(series_df):
    """One row per abnormality (ROI): its full-image series UID and its ROI series UID."""
    df = series_df[series_df["PatientID"].str.startswith("Mass-")].copy()
    full = {}
    for _, r in df.iterrows():
        if FULL_PID.match(r["PatientID"]):
            full[r["PatientID"]] = r["SeriesInstanceUID"]
    rows = []
    for _, r in df.iterrows():
        m = ROI_PID.match(r["PatientID"])
        if not m:
            continue
        full_pid = m.group(1)
        if full_pid not in full:
            continue
        parts = full_pid.split("_")  # Mass-Training, P, 00001, LEFT, CC
        rows.append(dict(
            roi_id=r["PatientID"],
            full_id=full_pid,
            patient=f"P_{parts[2]}",
            split=m.group(2),
            side=m.group(3),
            view=m.group(4),
            full_uid=full[full_pid],
            roi_uid=r["SeriesInstanceUID"],
        ))
    # an ROI pid can appear in two series (cropped / mask stored separately) -> keep all uids
    t = pd.DataFrame(rows)
    t = t.groupby(["roi_id", "full_id", "patient", "split", "side", "view", "full_uid"], as_index=False)["roi_uid"].agg(list)
    return t.sort_values("roi_id").reset_index(drop=True)


def sample_cases(cases, max_cases, seed=42):
    """Pick whole patients at random until ~max_cases ROIs, so folds stay patient-level."""
    pats = sorted(cases["patient"].unique())
    random.Random(seed).shuffle(pats)
    keep, n = [], 0
    for p in pats:
        k = int((cases["patient"] == p).sum())
        if n + k > max_cases and n > 0:
            continue
        keep.append(p)
        n += k
        if n >= max_cases:
            break
    return cases[cases["patient"].isin(keep)].reset_index(drop=True)


# ---------- image processing ----------
def to_uint8(img):
    """Robust 1-99.5 percentile windowing of a full mammogram to uint8."""
    img = img.astype(np.float32)
    nz = img[img > 0]
    lo, hi = (np.percentile(nz, 1), np.percentile(nz, 99.5)) if nz.size else (img.min(), img.max())
    img = np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)
    return (img * 255).astype(np.uint8)


def is_mask(arr):
    """CBIS ROI series hold a binary mask and a cropped image; the mask has <=2 values."""
    return np.unique(arr[:: max(1, arr.shape[0] // 256), :: max(1, arr.shape[1] // 256)]).size <= 2


def pick_mask(arrays, full_shape):
    """Among the arrays of an ROI series, return the binary mask (resized to the full image if needed)."""
    cands = [a for a in arrays if is_mask(a)]
    if not cands:
        return None
    m = max(cands, key=lambda a: a.size)
    m = (m > 0).astype(np.uint8)
    if m.shape != tuple(full_shape):  # a few CBIS masks are off by some pixels
        m = cv2.resize(m, (full_shape[1], full_shape[0]), interpolation=cv2.INTER_NEAREST)
    return m


def roi_crop(img, mask, size=256, margin=0.2):
    """Square crop around the mask bbox with a relative margin, resized to size x size."""
    ys, xs = np.where(mask > 0)
    if ys.size == 0:
        return None, None, None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    side = int(max(y1 - y0, x1 - x0) * (1 + 2 * margin))
    side = max(side, 64)
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    H, W = mask.shape
    top, left = cy - side // 2, cx - side // 2
    pad = max(0, -top, -left, top + side - H, left + side - W)
    if pad:
        img = np.pad(img, pad, mode="constant")
        mask = np.pad(mask, pad, mode="constant")
        top, left = top + pad, left + pad
    ic = img[top: top + side, left: left + side]
    mc = mask[top: top + side, left: left + side]
    ic = cv2.resize(ic, (size, size), interpolation=cv2.INTER_AREA)
    mc = cv2.resize(mc, (size, size), interpolation=cv2.INTER_NEAREST)
    return ic, (mc > 0).astype(np.uint8) * 255, dict(bbox_h=int(y1 - y0), bbox_w=int(x1 - x0), crop_side=side)


def jit_crop(img, mask, rid, size=CROP_SIZE):
    ys, xs = np.where(mask > 0)
    if ys.size == 0:
        return None, None, None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    L = max(y1 - y0, x1 - x0)
    rng = np.random.default_rng(int(hashlib.md5(rid.encode()).hexdigest()[:8], 16))
    scale = rng.uniform(SCALE_LO, SCALE_HI)
    side = max(int(L * scale), 64)
    slack_y, slack_x = (side - (y1 - y0)) / 2, (side - (x1 - x0)) / 2
    dy, dx = rng.uniform(-SHIFT, SHIFT, 2)
    cy = int(round((y0 + y1) / 2 + dy * slack_y)); cx = int(round((x0 + x1) / 2 + dx * slack_x))
    H, W = mask.shape
    top, left = cy - side // 2, cx - side // 2
    pad = max(0, -top, -left, top + side - H, left + side - W)
    if pad:
        img = np.pad(img, pad, mode='constant'); mask = np.pad(mask, pad, mode='constant'); top += pad; left += pad
    ic = cv2.resize(img[top:top + side, left:left + side], (size, size), interpolation=cv2.INTER_AREA)
    mc = cv2.resize(mask[top:top + side, left:left + side].astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST)
    mc = (mc > 0).astype(np.uint8) * 255
    return ic, mc, dict(bbox_h=int(y1 - y0), bbox_w=int(x1 - x0), crop_side=side, scale=round(float(scale), 4),
                        shift_y=round(float(dy), 4), shift_x=round(float(dx), 4), padded=bool(pad),
                        mask_frac=round(float((mc > 0).mean()), 4))


In [ ]:
# 3 · CBIS-DDSM: same 400 ROIs, re-downloaded and re-cropped (resumable)
from tqdm.auto import tqdm
subset = pd.read_csv(f'{OLD_CBIS}/cases_subset.csv'); subset['roi_uid'] = subset['roi_uid'].apply(json.loads)
old_folds = pd.read_csv(f'{OLD_CBIS}/folds.csv')[['roi_id', 'fold']]
assert set(subset.roi_id) == set(old_folds.roi_id), 'subset and folds differ'
def read_dicoms(files):
    out = []
    for f in files:
        try: out.append(pydicom.dcmread(f).pixel_array)
        except Exception as e: print('   unreadable', os.path.basename(f), e)
    return out
meta_csv = f'{NEW_CBIS}/metadata.csv'
meta = pd.read_csv(meta_csv).to_dict('records') if os.path.exists(meta_csv) else []
done = {m['roi_id'] for m in meta}; failed = []
for full_id, grp in tqdm(subset.groupby('full_id'), desc='images'):
    todo = grp[~grp.roi_id.isin(done)]
    if todo.empty: continue
    arrs = read_dicoms(download_series(grp.full_uid.iloc[0], RAW_TMP))
    if not arrs:
        failed += list(todo.roi_id); shutil.rmtree(RAW_TMP, ignore_errors=True); os.makedirs(RAW_TMP); continue
    full = max(arrs, key=lambda a: a.size); full8 = to_uint8(full)
    for _, r in todo.iterrows():
        a = []
        for uid in r.roi_uid: a += read_dicoms(download_series(uid, RAW_TMP))
        mask = pick_mask(a, full.shape)
        ic, mc, info = jit_crop(full8, mask, r.roi_id) if mask is not None else (None, None, None)
        if ic is None: failed.append(r.roi_id); continue
        cv2.imwrite(f'{NEW_CBIS}/images/{r.roi_id}.png', ic); cv2.imwrite(f'{NEW_CBIS}/masks/{r.roi_id}.png', mc)
        meta.append(dict(roi_id=r.roi_id, patient=r.patient, split=r.split, side=r.side, view=r.view, **info)); done.add(r.roi_id)
    pd.DataFrame(meta).to_csv(meta_csv, index=False)
    shutil.rmtree(RAW_TMP, ignore_errors=True); os.makedirs(RAW_TMP)
print(f'CBIS done: {len(done)} / {len(subset)} | failed: {failed[:10]}')
m = pd.DataFrame(meta).merge(old_folds, on='roi_id')
m = m.set_index('roi_id').loc[old_folds.roi_id].reset_index()       # same row order as the old folds.csv
m.to_csv(f'{NEW_CBIS}/folds.csv', index=False)
print(m.groupby('fold').agg(rois=('roi_id', 'size'), patients=('patient', 'nunique')).T.to_string())
print(m[['scale', 'mask_frac']].describe().round(3).to_string())


In [ ]:
# 4 · DMID: same masses (same ids as before), re-cropped with the same jitter rule
import subprocess, requests, glob, tifffile
from PIL import Image
RAW, LOCAL = '/content/dmid_raw', '/content/dmid'
os.makedirs(RAW, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
art = requests.get('https://api.figshare.com/v2/articles/24522883', timeout=60).json()
for f in art['files']:
    if not any(k in f['name'] for k in ('TIFF', 'ROI', 'Metadata')): continue
    dst = f"{RAW}/{f['name']}"
    if not os.path.exists(dst) or os.path.getsize(dst) != f['size']:
        with requests.get(f['download_url'], stream=True, timeout=900) as r:
            r.raise_for_status()
            with open(dst, 'wb') as o:
                for ch in r.iter_content(1 << 20): o.write(ch)
    if dst.endswith('.zip') and not os.path.isdir(f"{LOCAL}/{f['name'][:-4]}"):
        subprocess.run(['unzip', '-q', '-o', dst, '-d', LOCAL], check=False)
def read_tif(p):
    try: a = tifffile.imread(p)
    except Exception: a = np.array(Image.open(p))
    a = np.asarray(a)
    if a.ndim == 3 and a.shape[0] in (1, 3, 4) and a.shape[-1] not in (1, 3, 4): a = np.moveaxis(a, 0, -1)
    if a.ndim == 3: a = a[..., :3].mean(-1) if a.shape[-1] >= 3 else a[..., 0]
    return a
info_path = glob.glob(f'{LOCAL}/**/Info.txt', recursive=True)[0]
rows = []
for line in open(info_path, errors='ignore'):
    p = line.split()
    if len(p) >= 3 and p[0].upper().startswith('IMG'):
        def _f(s):
            try: return float(s)
            except ValueError: return np.nan
        rows.append(dict(img=p[0].upper(), tissue=p[1], cls=p[2], sev=p[3] if len(p) > 3 else None,
                         x=_f(p[4]) if len(p) > 6 else np.nan, y=_f(p[5]) if len(p) > 6 else np.nan, r=_f(p[6]) if len(p) > 6 else np.nan))
info = pd.DataFrame(rows)
IMG_DIR = glob.glob(f'{LOCAL}/**/TIFF Images', recursive=True)[0]
ROI_DIR = glob.glob(f'{LOCAL}/**/ROI Masks', recursive=True)[0]
mass = info[info.cls.isin(['CIRC', 'SPIC', 'MISC']) & info.x.notna() & info.y.notna()].copy()
mass['r'] = mass.r.fillna(50).clip(lower=20)
mass = mass.reset_index(drop=True)
def comps(m): return cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
def pick(n, lab, st, cen, x, y, r):
    keep = []
    for k in range(1, n):
        x0, y0, w, h, _ = st[k]
        dx = max(x0 - x, 0, x - (x0 + w - 1)); dy = max(y0 - y, 0, y - (y0 + h - 1))
        if dx * dx + dy * dy <= (1.5 * r) ** 2: keep.append(k)
    return keep
rec = []
for img_id, grp in tqdm(list(mass.groupby('img', sort=True)), desc='DMID images'):
    f = f'{ROI_DIR}/{img_id}.tif'
    if not os.path.exists(f): continue
    m = read_tif(f) > 0
    im = read_tif(f'{IMG_DIR}/{img_id}.tif').astype(np.float32)
    if im.shape != m.shape:
        m = cv2.resize(m.astype(np.uint8), (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
    im8 = to_uint8(im); c = comps(m)
    for i, r in grp.iterrows():
        keep = pick(*c, r.x, r.y, r.r)                      # y measured from the top (verified in 03b2)
        if not keep: continue
        rid = f'{img_id}_{i}'
        ic, mc, mi = jit_crop(im8, np.isin(c[1], keep), rid)
        cv2.imwrite(f'{NEW_DMID}/images/{rid}.png', ic); cv2.imwrite(f'{NEW_DMID}/masks/{rid}.png', mc)
        rec.append(dict(roi_id=rid, img=img_id, patient=img_id, cls=r.cls, sev=r.sev, tissue=r.tissue, n_comp=len(keep), **mi))
dm = pd.DataFrame(rec); dm.to_csv(f'{NEW_DMID}/metadata.csv', index=False)
old = pd.read_csv(OLD_DMID_DEDUP)
print('DMID crops:', len(dm), '| same ids as before:', set(dm.roi_id) == set(old.roi_id))


In [ ]:
# 5 · No-network baselines on the NEW crops (go / no-go) + QC grid
import matplotlib.pyplot as plt
def load(d, ids, mask):
    return np.stack([(cv2.imread(f'{d}/{"masks" if mask else "images"}/{r}.png', 0) > 127) if mask else cv2.imread(f'{d}/images/{r}.png', 0) for r in ids])
F = pd.read_csv(f'{NEW_CBIS}/folds.csv'); YC = load(NEW_CBIS, F.roi_id, True)
dm = pd.read_csv(f'{NEW_DMID}/metadata.csv'); keep = pd.read_csv(OLD_DMID_DEDUP); keep = set(keep.roi_id[keep.keep == True])
dm = dm[dm.roi_id.isin(keep)].reset_index(drop=True); YD = load(NEW_DMID, dm.roi_id, True)
def dice(Y, P):
    tp = (Y & P).sum((1, 2)); s = Y.sum((1, 2)) + P.sum((1, 2))
    return np.where(s == 0, 1.0, 2 * tp / np.maximum(s, 1))
def disc(area, n=256):
    r = np.sqrt(area / np.pi); yy, xx = np.mgrid[:n, :n]
    return (yy - (n - 1) / 2) ** 2 + (xx - (n - 1) / 2) ** 2 <= r ** 2
rows = []
for k in range(5):
    f = F.fold.values; te = f == k; tr = (f != k) & (f != (k + 1) % 5)
    shapes = {'mean_mask': YC[tr].mean(0) > 0.5, 'centred_disc': disc(YC[tr].sum((1, 2)).mean())}
    for nm, sh in shapes.items():
        rows.append(dict(baseline=nm, fold=k, dataset='CBIS', dice=float(dice(YC[te], sh[None]).mean())))
        rows.append(dict(baseline=nm, fold=k, dataset='DMID', dice=float(dice(YD, sh[None]).mean())))
b = pd.DataFrame(rows); b.to_csv(f'{PROJECT}/results/p2_jit_baselines.csv', index=False)
print(b.groupby(['baseline', 'dataset']).dice.agg(['mean', 'std']).round(3).to_string())
print('Old crops for comparison: centred disc 0.861 (CBIS), 0.806 (DMID).')
fig, ax = plt.subplots(2, 6, figsize=(15, 5.4))
for a, (d, ids) in zip(ax.ravel(), [(NEW_CBIS, r) for r in F.roi_id.sample(6, random_state=0)] + [(NEW_DMID, r) for r in dm.roi_id.sample(6, random_state=0)]):
    im = cv2.imread(f'{d}/images/{ids}.png', 0); mk = cv2.imread(f'{d}/masks/{ids}.png', 0)
    rgb = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB); cs, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(rgb, cs, -1, (255, 60, 60), 2); a.imshow(rgb); a.set_title(ids, fontsize=7); a.axis('off')
plt.tight_layout(); plt.savefig(f'{PROJECT}/results/p2_jit_qc.png', dpi=110); plt.show()
